## Observações

- A coluna id tem valores não numéricos e mal formatados, e.g., `12_1`, `127_1`, `30_5`
- Não há duplicatas
- Há pouquissímas colunas com quantidade significativa de valores ausentes
- Não há amostras extremamente incompletas
- Não há colunas extremamente constantes
- O cromossomo X apresenta a maior quantidade de loci esparsos, com média de 31% de valores ausentes.

## Decisões

- Dropar colunas com mais de 5% de valores ausentes
- Imputar ausentes pelo genótipo mais frequente

In [ ]:
import pandas as pd

from covid.eda.quality import missing_summary_by_chromosome
from covid.experiments.shared import feature
from covid.paths import INTERIM_TRAIN_DATA_PATH, RAW_DATA_PATH

full_data = pd.read_csv(RAW_DATA_PATH, dtype={feature.ID: str})
full_data.shape

## Checking Inconsistencies

In [ ]:
full_data[~full_data[feature.ID].str.isdigit()][feature.ID]

In [ ]:
full_data.duplicated().sum()

## Checking sparse columns

In [ ]:
train_data = pd.read_csv(INTERIM_TRAIN_DATA_PATH, dtype={feature.ID: str})
train_data.shape

In [ ]:
train_data.isna().sum().sort_values(ascending=False)

In [ ]:
missing_percent = train_data.isna().mean().mul(100)
missing_percent.plot.hist(
    bins=50,
    title="Distribution of missing values by column",
    xlabel="Missing values (%)",
    ylabel="Number of columns",
)

In [ ]:
import numpy as np

upper_percent = 10
missing_percent.plot.hist(
    title="Distribution of missing values by column (zoomed in)",
    xlabel="Missing values (%)",
    ylabel="Number of columns",
    bins=upper_percent * 10,
    xlim=(0, upper_percent),
    xticks=np.arange(0, upper_percent),
)

In [ ]:
quantiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
quantile_analysis = pd.DataFrame(
    {
        "quantile": quantiles,
        "percent": missing_percent.quantile(quantiles).round(2).values,
        "count": train_data.isna().sum().quantile(quantiles).astype(int).values
    },
)
quantile_analysis

In [ ]:
missing_ratio = train_data.isna().mean()

threshold = 0.05
percentile = missing_ratio.le(threshold).mean() * 100

print(f"Colunas with up to {threshold:.0%} missing values: {percentile:.1f} percentile")

In [ ]:
import seaborn as sns

top_missing = train_data.isna().sum().sort_values(ascending=False).head(20)
sns.barplot(data=top_missing, orient="h")

In [ ]:
import missingno as msno

cols_for_matrix_plot = train_data.isna().sum().sort_values(ascending=False).head(300)
msno.matrix(train_data[cols_for_matrix_plot.index])

## Checking Variance

In [ ]:
variances = train_data[feature.get_loci_columns(train_data)].var()
variances.sort_values(ascending=True).head(20)

In [ ]:
variances.hist(bins=50)

In [ ]:
variances.describe(percentiles=[0.001, 0.01, 0.05, 0.10, 0.25]).round(3)

## Missing by chromosome

In [ ]:
missing_summary_by_chromosome(loci_data=feature.get_loci_data(train_data))

## Checking sparse samples

In [ ]:
missing_by_row = train_data.isna().sum(axis=1)

missing_by_row.describe()

In [ ]:
missing_by_row.plot(
    kind="hist",
    bins=30,
    title="Number of missing features by sample",
    xlabel="Number of missing features",
    ylabel="Number of samples",
)

In [ ]:
msno.matrix(train_data.T)

In [ ]:
missing_ratio_by_row = train_data.isna().mean(axis=1)
missing_ratio_by_row.describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
sparse_samples = missing_ratio_by_row.sort_values(ascending=False).index
train_data.loc[sparse_samples].head(10)